In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("workspace.bronze.inventory")

df = (
    df
    .withColumn("quantity_available", F.col("quantity_available").cast("int"))
    .withColumn("quantity_reserved", F.col("quantity_reserved").cast("int"))
    .withColumn("updated_at", F.to_timestamp("updated_at", "dd-MM-yyyy HH:mm"))
)



In [0]:
window = (
    Window
    .partitionBy("product_id", "warehouse_id")
    .orderBy(F.col("updated_at").desc())
)

silver_inventory = (
    df
    .withColumn("rn", F.row_number().over(window))
    .filter("rn = 1")
    .drop("rn")
    .filter(F.col("quantity_available") >= 0)
)

(
    silver_inventory.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.inventory")
)



In [0]:
display(silver_inventory)